# 01. Preparación de datos y particiones

**NewCodeCreds** · Edward Alexander Gómez Ispanel, Jocelyn Abigail De León Ardón y Brazil Abraham
Batres Cruz - Maestría en Data Science - Investigación en Data Science - Noviembre 2026

Datos: CredData (Samsung), commit 86a9182: 67,896 etiquetas de líneas candidatas (T, F o X)
en 337 repositorios públicos, con los archivos de código en `CredData/data`. Este es el
primero de cuatro notebooks; los siguientes usan la tabla que se guarda aquí.

**Pregunta vigente:** En código de repositorios públicos, ¿cómo cambia el desempeño de
detectores de credenciales (reglas, aprendizaje automático y modelos de lenguaje) cuando
se evalúan en repositorios o tipos de archivo no vistos frente a particiones aleatorias?

Este notebook construye la tabla de análisis (una fila = una línea candidata) a partir de
`CredData/meta` y `CredData/data` (commit 86a9182), aplica las decisiones de la Actividad 04
y asigna cada línea a los *folds* de las tres particiones:

- aleatoria sin deduplicar (para comparar con Biringa y Kul, 2025),
- aleatoria deduplicada (baseline de las hipótesis),
- por repositorio agrupando por organización (StratifiedGroupKFold),
- por tipo de archivo (se deja fuera un lenguaje a la vez).

## 0. Configuración

Rutas relativas a esta carpeta (`notebooks/`): CredData se espera en `../../CredData`, al lado
de la carpeta del repositorio. La semilla (42) y el repositorio dominante (OpenSSL) están en
`ids_utils.py`, que comparten los cuatro notebooks.

In [1]:
import binascii
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

sys.path.append(".")
from ids_utils import (REPO_DOMINANTE, SEMILLA, grupo_tipo_archivo,
                       lineas_de_archivo, mapa_organizaciones)

CRED = Path("../../CredData")
SALIDA = Path("../datos_procesados")
RES = Path("../resultados")
SALIDA.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)
N_PLIEGUES = 5
LENGUAJES_EVALUADOS = ["Go", "Python", "JavaScript", "Java", "Ruby", "C#", "TypeScript", "PHP"]
print("Semilla:", SEMILLA, "| CredData encontrado:", (CRED / "meta").exists())

Semilla: 42 | CredData encontrado: True


## 1. Cargar las etiquetas

In [2]:
meta = pd.concat([pd.read_csv(p, dtype=str) for p in sorted((CRED / "meta").glob("*.csv"))],
                 ignore_index=True)
for c in ["LineStart", "LineEnd"]:
    meta[c] = meta[c].astype(int)
meta["Domain"] = meta["Domain"].str.replace("Github", "GitHub")
print("Filas:", len(meta), "| repositorios:", meta.RepoName.nunique(), "| archivos:", meta.FileID.nunique())
print(meta.GroundTruth.value_counts())

Filas: 67896 | repositorios: 337 | archivos: 10794
GroundTruth
F    48061
T    15753
X     4082
Name: count, dtype: int64


## 2. Variables derivadas

### 2.1 Organización, extensión y carpeta

In [3]:
orgs = mapa_organizaciones(CRED / "snapshot.json")
meta = meta.merge(orgs[["RepoName", "organizacion"]], on="RepoName", how="left")
print("Repos sin organización:", meta.organizacion.isna().sum(), "| organizaciones:", meta.organizacion.nunique())

ruta = meta.FilePath.str.split("/")
meta["carpeta"] = ruta.str[2]
meta["extension"] = meta.FilePath.str.extract(r"(\.[^./]+)$", expand=False).str.lower().fillna("")
meta["tipo_archivo"] = meta.extension.map(grupo_tipo_archivo)
meta["es_test"] = meta.carpeta.str.lower().str.contains("test").astype(int)
meta["y"] = (meta.GroundTruth == "T").astype(int)

Repos sin organización: 0 | organizaciones: 307


### 2.2 Texto de la línea

Las líneas se numeran con la misma regla que CredData (`review_data.py`): `\r\n` y `\r` pasan
a `\n` y se corta solo por `\n`. Una versión anterior usaba `str.splitlines()`, que también
corta en caracteres como `\x0c` (salto de página) o `\u2028`; en 5 archivos eso desalineaba el
número de línea y 35 líneas candidatas (8 T) recibían el texto de otra línea.

In [4]:
cache = {}


def leer_lineas(path):
    if path not in cache:
        try:
            cache[path] = lineas_de_archivo((CRED / path).read_text(encoding="utf-8", errors="ignore"))
        except FileNotFoundError:
            cache[path] = None
    return cache[path]


textos = []
for fp, ls, le in zip(meta.FilePath, meta.LineStart, meta.LineEnd):
    lineas = leer_lineas(fp)
    if lineas is None or ls < 1 or ls > len(lineas):
        textos.append(None)
        continue
    textos.append("\n".join(lineas[ls - 1:le])[:1000])
meta["texto"] = textos
cache.clear()
print("Líneas sin texto recuperado:", meta.texto.isna().sum())

Líneas sin texto recuperado: 0


## 3. Limpieza y deduplicación

### 3.1 Una fila por línea de código

CredData etiqueta valores, no líneas: una línea con dos valores tiene dos filas, y hay filas
duplicadas exactas. Se deja una fila por línea (archivo, LineStart, LineEnd). Si alguna de
sus filas es T, la línea es T: es la convención del benchmark y la unidad de análisis del
estudio. Así se resuelven las líneas que tienen un valor T y otro F o X.

In [5]:
n0 = len(meta)
meta = meta.dropna(subset=["texto"])
meta = meta.drop_duplicates(subset=["FilePath", "LineStart", "LineEnd", "GroundTruth"])
etiquetas_por_linea = meta.groupby(["FilePath", "LineStart", "LineEnd"]).GroundTruth.agg(set)
mixtas = etiquetas_por_linea[etiquetas_por_linea.map(len) > 1]
print("Líneas con etiquetas distintas en la misma línea:", len(mixtas),
      "| con T y no T:", int(mixtas.map(lambda e: "T" in e).sum()))
# Misma línea con etiquetas distintas: se queda la etiqueta T si existe (convención del benchmark)
meta = (meta.sort_values("y", ascending=False)
            .drop_duplicates(subset=["FilePath", "LineStart", "LineEnd"]))
meta = meta.sort_values(["RepoName", "FilePath", "LineStart"]).reset_index(drop=True)
meta["id_linea"] = np.arange(len(meta))
print(f"Filas: {n0} -> {len(meta)} tras quitar duplicados exactos y conflictos en la misma línea")

Líneas con etiquetas distintas en la misma línea: 351 | con T y no T: 253
Filas: 67896 -> 64797 tras quitar duplicados exactos y conflictos en la misma línea


**Qué salió.** De las 67,896 filas de `meta/` quedan 64,797 líneas, con 15,280 T. Hay 351 líneas con
etiquetas distintas en la misma línea, y en 253 de ellas una es T, así que quedan como T. Estas
cifras reemplazan las de la Actividad 04 (420 líneas con etiquetas distintas y 259 textos
contradictorios), que se calcularon antes de unir las filas de una misma línea y con la lectura
anterior del texto.

### 3.2 Deduplicación por texto normalizado

Se conserva una línea por texto (espacios colapsados). Después de 3.1 ya no queda ningún
texto con etiquetas mezcladas (T en una línea y F o X en otra con el mismo texto), así que
no hay que elegir etiqueta mayoritaria ni descartar empates. El `assert` lo verifica: si
cambiara la versión de CredData y aparecieran textos contradictorios, el notebook se detiene.

In [6]:
meta["texto_norm"] = meta.texto.str.strip().str.replace(r"\s+", " ", regex=True)
conteo = meta.groupby("texto_norm").y.agg(["mean", "size"])
empates = conteo[(conteo["mean"] == 0.5)].index
contradictorios = conteo[(conteo["mean"] > 0) & (conteo["mean"] < 1)]
print("Textos con etiquetas contradictorias:", len(contradictorios), "| empates descartados:", len(empates))
assert len(contradictorios) == 0 and len(empates) == 0, "Hay textos con etiquetas mezcladas: revisar la regla"

etiqueta_mayoritaria = (conteo["mean"] > 0.5).astype(int)
meta["dedup"] = 0
primeras = meta[~meta.texto_norm.isin(empates)].drop_duplicates("texto_norm").index
meta.loc[primeras, "dedup"] = 1
meta["y_dedup"] = meta.texto_norm.map(etiqueta_mayoritaria)
d = meta[meta.dedup == 1]
print(f"Conjunto deduplicado: {len(d)} líneas, {d.y_dedup.sum()} T ({d.y_dedup.mean():.1%})")

# Qué quita la deduplicación: casi todo son negativos
quitadas = meta[meta.dedup == 0]
print(f"Quitadas: {len(quitadas)} líneas, {quitadas.y.sum()} T y {(quitadas.y == 0).sum()} no T")
t_cruzan = (meta[meta.y == 1].groupby("texto_norm").RepoName.nunique() > 1)
print("Textos T que aparecen en más de un repositorio:", int(t_cruzan.sum()),
      "| filas T con esos textos:", int(meta[(meta.y == 1) & meta.texto_norm.isin(t_cruzan[t_cruzan].index)].shape[0]))

Textos con etiquetas contradictorias: 0 | empates descartados: 0
Conjunto deduplicado: 49629 líneas, 15205 T (30.6%)
Quitadas: 15168 líneas, 75 T y 15093 no T
Textos T que aparecen en más de un repositorio: 4 | filas T con esos textos: 8


**Qué salió.** El conjunto deduplicado tiene 49,629 líneas, con 15,205 T (30.6%). La deduplicación
quita 15,168 líneas: 75 T y 15,093 no T. Es decir, afecta casi solo a los **negativos** (plantillas,
configuraciones y líneas de ejemplo repetidas), no a las credenciales. Solo 4 textos T aparecen en
más de un repositorio (8 filas T). La cifra de la Actividad 04, "1,035 casos T repiten el texto de
línea de otro caso T de otro repositorio", no se reproduce con el texto de la línea y queda
reemplazada por esta. Tampoco quedan textos con etiquetas mezcladas, como verifica el `assert`.

## 4. Particiones

### 4.1 Asignación de *folds*

Las asignaciones de la partición aleatoria deduplicada y de la partición por organización se
fijan por archivo (`resultados/asignacion_pliegues_*.csv`) y se leen si existen. Motivo:
`StratifiedGroupKFold` con la misma semilla da otra asignación en otra versión de scikit-learn
(entre la versión de la VM y la 1.8.0 del Mac coincide solo el 36% de las líneas), y
`StratifiedKFold` cambia toda la asignación si entra o sale una sola línea del conjunto. Los
archivos guardan la asignación de la primera versión de la tabla, con la que también se
corrió GraphCodeBERT, así que los tres detectores comparten *folds*. Solo se recalculan si
los archivos no existen.

Todas las predicciones se generan fuera de muestra (out-of-fold): cada línea recibe una
predicción de un modelo que no la vio. Lo que cambia entre particiones es si el modelo vio
o no su repositorio (organización) o su tipo de archivo.

In [7]:
rng = np.random.RandomState(SEMILLA)

# 4a. Aleatoria sin deduplicar (todas las líneas)
meta["pliegue_aleatorio_crudo"] = -1
skf = StratifiedKFold(n_splits=N_PLIEGUES, shuffle=True, random_state=SEMILLA)
for k, (_, te) in enumerate(skf.split(meta, meta.y)):
    meta.loc[te, "pliegue_aleatorio_crudo"] = k

# 4b. Aleatoria deduplicada (asignación fija por línea)
idx_d = meta.index[meta.dedup == 1]
CLAVE = ["FilePath", "LineStart", "LineEnd"]
ASIG_ALE = RES / "asignacion_pliegues_aleatorio.csv"
meta["pliegue_aleatorio"] = -1
if ASIG_ALE.exists():
    asig = pd.read_csv(ASIG_ALE)
    fijo = meta.loc[idx_d, CLAVE].merge(asig, on=CLAVE, how="left").pliegue.to_numpy(dtype=float, copy=True)
    nuevas = np.isnan(fijo)
    # Líneas que no estaban en el conjunto deduplicado de la asignación guardada: pliegue por
    # CRC32 de su clave, determinista y sin depender de la librería
    claves = meta.loc[idx_d[nuevas], CLAVE].astype(str).agg("|".join, axis=1)
    fijo[nuevas] = [binascii.crc32(c.encode()) % N_PLIEGUES for c in claves]
    meta.loc[idx_d, "pliegue_aleatorio"] = fijo.astype(int)
    print("Pliegue aleatorio leído de", ASIG_ALE.name, "| líneas sin asignación previa:", int(nuevas.sum()))
else:
    for k, (_, te) in enumerate(skf.split(idx_d, meta.loc[idx_d, "y_dedup"])):
        meta.loc[idx_d[te], "pliegue_aleatorio"] = k
    meta.loc[idx_d, CLAVE + ["pliegue_aleatorio"]].rename(columns={"pliegue_aleatorio": "pliegue"}).to_csv(ASIG_ALE, index=False)

# 4c. Por repositorio, agrupando por organización y estratificando por la etiqueta
ASIG_ORG = RES / "asignacion_pliegues_organizacion.csv"
meta["pliegue_organizacion"] = -1
if ASIG_ORG.exists():
    asig = pd.read_csv(ASIG_ORG).set_index("organizacion").pliegue
    assert meta.loc[idx_d, "organizacion"].isin(asig.index).all(), "Hay organizaciones sin pliegue asignado"
    meta.loc[idx_d, "pliegue_organizacion"] = meta.loc[idx_d, "organizacion"].map(asig).astype(int)
    print("Pliegue por organización leído de", ASIG_ORG.name, "|", len(asig), "organizaciones")
else:
    sgkf = StratifiedGroupKFold(n_splits=N_PLIEGUES, shuffle=True, random_state=SEMILLA)
    for k, (_, te) in enumerate(sgkf.split(idx_d, meta.loc[idx_d, "y_dedup"], meta.loc[idx_d, "organizacion"])):
        meta.loc[idx_d[te], "pliegue_organizacion"] = k
    (meta.loc[idx_d].groupby("organizacion").pliegue_organizacion.first().rename("pliegue")
         .to_csv(ASIG_ORG))

# 4d. Por tipo de archivo: el pliegue es el lenguaje (solo los ocho con casos suficientes)
meta["pliegue_tipo"] = np.where((meta.dedup == 1) & meta.tipo_archivo.isin(LENGUAJES_EVALUADOS),
                                 meta.tipo_archivo, "")

Pliegue aleatorio leído de asignacion_pliegues_aleatorio.csv | líneas sin asignación previa: 6
Pliegue por organización leído de asignacion_pliegues_organizacion.csv | 307 organizaciones


### 4.2 Resumen de los *folds*

In [8]:
d = meta[meta.dedup == 1]
resumen_pliegues = (d.groupby("pliegue_organizacion")
                      .agg(lineas=("y_dedup", "size"), positivos=("y_dedup", "sum"),
                           organizaciones=("organizacion", "nunique"), repos=("RepoName", "nunique"),
                           contiene_openssl=("RepoName", lambda s: int(REPO_DOMINANTE in set(s)))))
resumen_pliegues["prop_T"] = resumen_pliegues.positivos / resumen_pliegues.lineas
print(resumen_pliegues)
resumen_pliegues.to_csv(RES / "pliegues_organizacion.csv")

resumen_tipo = (d[d.pliegue_tipo != ""].groupby("pliegue_tipo")
                  .agg(lineas=("y_dedup", "size"), positivos=("y_dedup", "sum"), repos=("RepoName", "nunique")))
print(resumen_tipo)
resumen_tipo.to_csv(RES / "pliegues_tipo_archivo.csv")

# Verificación: ninguna organización aparece en dos pliegues
assert (d.groupby("organizacion").pliegue_organizacion.nunique() == 1).all()

                      lineas  positivos  organizaciones  repos  \
pliegue_organizacion                                             
0                      16492       7010              64     72   
1                       9721       2894              73     78   
2                      10337       2241              50     58   
3                       7752       1794              55     61   
4                       5327       1266              65     68   

                      contiene_openssl    prop_T  
pliegue_organizacion                              
0                                    1  0.425055  
1                                    0  0.297706  
2                                    0  0.216794  
3                                    0  0.231424  
4                                    0  0.237657  
              lineas  positivos  repos
pliegue_tipo                          
C#              1043        243     19
Go              4871       1429     57
Java            1402    

**Qué salió.** Los *folds* por organización tienen entre 5,327 y 16,492 líneas. El *fold* 0 contiene
a OpenSSL y por eso es el más grande y el de mayor proporción de T (42.5%, frente a 22% a 30% en los
demás). Las 307 organizaciones caen cada una en un solo *fold*. Los ocho lenguajes suman 18,695
líneas: Go es el de más casos T (1,429) y PHP el de menos (150). Las asignaciones se leyeron de
`resultados/asignacion_pliegues_*.csv`. Solo las 6 líneas que entraron al conjunto deduplicado con
la lectura corregida recibieron un *fold* nuevo.

## 5. Guardar la tabla de análisis

El archivo con el texto de las líneas se queda en local (`datos_procesados/`), porque
contiene código con licencia de cada proyecto. El CSV que se comparte se genera en el
notebook 02 sin la columna de texto.

In [9]:
cols = ["id_linea", "Id", "FileID", "RepoName", "organizacion", "FilePath", "LineStart", "LineEnd",
        "GroundTruth", "Category", "carpeta", "es_test", "extension", "tipo_archivo", "y", "dedup",
        "y_dedup", "pliegue_aleatorio_crudo", "pliegue_aleatorio", "pliegue_organizacion",
        "pliegue_tipo", "texto"]
meta[cols].to_parquet(SALIDA / "lineas.parquet", index=False)

resumen = {
    "filas_meta": n0, "filas_analisis": len(meta), "lineas_dedup": int(meta.dedup.sum()),
    "T_dedup": int(meta.loc[meta.dedup == 1, "y_dedup"].sum()),
    "lineas_etiquetas_mixtas": len(mixtas), "lineas_T_y_no_T": int(mixtas.map(lambda e: "T" in e).sum()),
    "textos_contradictorios": len(contradictorios), "empates_descartados": len(empates),
    "organizaciones": meta.organizacion.nunique(), "repositorios": meta.RepoName.nunique(),
}
pd.Series(resumen).to_csv(RES / "resumen_preparacion.csv", header=["valor"])
pd.Series(resumen)

filas_meta                 67896
filas_analisis             64797
lineas_dedup               49629
T_dedup                    15205
lineas_etiquetas_mixtas      351
lineas_T_y_no_T              253
textos_contradictorios         0
empates_descartados            0
organizaciones               307
repositorios                 337
dtype: int64